# RoSE + model routing — FigureQA factoid (first 400 charts)

Per question: the **kNN router** (built from the ChartQAPro pilot pool)
picks a model first — Gemma 3 4B, Qwen3.5-4B, or Qwen2.5-VL-7B — then the
question runs through the **full RoSE pipeline** with that model.

FigureQA is synthetic charts with yes/no questions; answers are scored by
the repo's `scoring.py` exact match ("Yes"/"No" are factoid-style gold).
Source: the `figureqa` subset of `HuggingFaceM4/the_cauldron` (parquet,
streamed — no bulk download). The run takes **one question from each of
the first `N_QUESTIONS` charts in stream order** — deterministic, no
sampling. Note the router's experience pool comes from ChartQAPro —
routing here measures how that experience transfers across datasets;
expect conservative (7B-heavy) decisions on unfamiliar question styles.

**Kaggle quota math (30 GPU-h/week).** 400 questions at the pilot's
latencies lands between **~12 h** (everything routed to Gemma, ~111 s/q)
and **~27 h** (everything to the 7B, ~240 s/q); the pilot's actual mix
(95% to 7B) means plan for the high end. That is MORE than one Kaggle
session (~9–12 h cap) and most of a weekly quota: run it across 2–3
sessions — the checkpoint lands after every question, so when a session
dies, upload `routed_<dataset>_checkpoint.json` as a Kaggle dataset, set
`RESUME_FROM`, and the next session continues where it stopped. The three
notebooks together are ~60–80 GPU-h ≈ **2–3 weeks of quota**; run them in
series, not in parallel. `DRY_RUN` is off — flip it to `True` for a
~20-minute 4-question smoke test if you want one before committing hours.

**Prerequisites**
1. Kaggle secret `HF_TOKEN` attached to this notebook (Gemma 3 is gated —
   accept the license on hf.co with the token's account).
2. `chartqapro/results/pilot/routing_pool.json` committed to the branch
   this notebook clones — or attached as a Kaggle dataset (the pool cell
   searches `/kaggle/input/*/`).
3. GPU: T4 x2 (Settings → Accelerator).


In [ ]:
# ── config ───────────────────────────────────────────────────────
DATASET_KEY = "figureqa"
N_QUESTIONS = 400         # first N questions in dataset order (see quota math above)
DRY_RUN     = False              # True = route everything, answer first 4 only (~20 min)
QUANT       = "nf4"              # the pilot spec
K, TAU      = None, None         # None = routing/config.py defaults (k=5, τ=0.6)
SESSION_BUDGET_H = 8.0           # stop cleanly after this many hours so a
                                 # "Save & Run All" version FINISHES inside
                                 # Kaggle's session cap and publishes the
                                 # checkpoint; None = run until done/killed

REPO_URL  = "https://github.com/ktahsinr/RoSE.git"
REPO_REF  = "feature/model-routing"

# A 400-question run spans MORE than one Kaggle session. When a session
# dies, download routed_figureqa_checkpoint.json from the Output pane,
# upload it as a Kaggle dataset, and point at it here — the next session
# resumes where the last one stopped.
RESUME_FROM = None

import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # Kaggle can't render the widgets
from pathlib import Path
import sys
try:
    sys.stdout.reconfigure(line_buffering=True)    # live logs in Kaggle's pane
except (AttributeError, ValueError):
    pass

ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"
RESULTS_DIR = WORK / "results" / "routed"          # published with the notebook
DATASET_JSON   = SCRATCH / "figureqa_factoid_routed.json"
DECISIONS_FILE = RESULTS_DIR / f"routing_decisions_{DATASET_KEY}.json"
OUT_FILE       = RESULTS_DIR / f"routed_{DATASET_KEY}.json"

for d in (SCRATCH, IMAGES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"dataset={DATASET_KEY}  n={N_QUESTIONS}  dry_run={DRY_RUN}  quant={QUANT}")


In [ ]:
# ── code + pinned libraries ──────────────────────────────────────
import subprocess
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    REPO_URL, str(CODE_DIR)], check=True)
else:
    # a kernel restart keeps /kaggle/temp — refresh so fixes pushed to the
    # branch mid-session are picked up instead of running stale code
    subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1",
                    "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "reset", "--hard",
                    "FETCH_HEAD"], check=True)
print("repo @", subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(CODE_DIR / "requirements-pinned.txt")], check=True)
# NOTE: transformers==4.51.3 (the sweep pin) supports Gemma 3. If
# Qwen/Qwen3.5-4B fails to load with an unknown-architecture error, install a
# newer transformers here for that session — the run's *_meta.json records the
# exact version either way, so the report stays honest.

sys.path.insert(0, str(CODE_DIR / "chartqapro"))
import transformers
print("transformers:", transformers.__version__)


In [ ]:
# ── HuggingFace auth + gated-access check (ALL routed models) ────
# A routed run may land on any of the three models, so access to every
# one is verified up front — better a failure here than 4 hours in.
#
# The check DOWNLOADS each model's config.json — the exact request the
# model load makes. (model_info() is NOT enough: a gated repo's metadata
# is public, so it "succeeds" even with no token and the run still 401s
# hours later at load time.)
from routing.config import MODELS, ESCALATION_ORDER

NO_TOKEN_HELP = (
    "→ No HF token reached this session:\n"
    "  1. hf.co → Settings → Access Tokens → create a READ token\n"
    "     (use the SAME account that accepted the Gemma 3 license)\n"
    "  2. Kaggle → Add-ons → Secrets → add it with label HF_TOKEN\n"
    "  3. TICK THE CHECKBOX attaching the secret to THIS notebook —\n"
    "     the attachment is per-notebook and defaults to off\n"
    "  then re-run this cell.\n")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"could not read Kaggle secret HF_TOKEN: {e}")

if token:
    from huggingface_hub import login
    login(token=token)
    print("HF login ok")
elif ON_KAGGLE:
    raise SystemExit("\nNO HF TOKEN.\n" + NO_TOKEN_HELP)

from huggingface_hub import hf_hub_download
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
for mk in ESCALATION_ORDER:
    spec = MODELS[mk]
    hf_id = spec["hf_id"]
    try:
        hf_hub_download(hf_id, "config.json", revision=spec["revision"])
        print(f"✓ access to {hf_id} verified (config.json downloaded)")
    except GatedRepoError:
        raise SystemExit(
            f"\nNO ACCESS to the gated model {hf_id}.\n"
            + (f"→ Your token logged in but the LICENSE is not accepted (or\n"
               f"  still pending) for ITS account: visit\n"
               f"  https://huggingface.co/{hf_id} logged in as the token's\n"
               "  account, accept the license, then re-run this cell.\n"
               if token else NO_TOKEN_HELP))
    except RepositoryNotFoundError:
        raise SystemExit(f"{hf_id} not found — with a gated repo this can also "
                         "mean the token has no access; check token + license.")


In [ ]:
# ── build the first-N FigureQA factoid set ───────────────────────
# Streams the `figureqa` subset of the_cauldron (100k rows, ~10 yes/no
# QAs per image). ONE question (the first) is taken per chart, from the
# FIRST N_QUESTIONS usable charts in stream order — deterministic, no
# sampling.
import json
from datasets import load_dataset

ds = load_dataset("HuggingFaceM4/the_cauldron", "figureqa",
                  split="train", streaming=True)

records = []
for row_idx, row in enumerate(ds):
    if len(records) >= N_QUESTIONS:
        break
    texts = row["texts"]
    if not texts or not row["images"]:
        continue
    qa = texts[0]                       # one QA per image, first in row order
    q = qa["user"].split("\n")[0].strip()          # drop "Answer yes or no."
    a = qa["assistant"].strip().rstrip(".")
    if a.lower() not in ("yes", "no"):
        continue
    path = IMAGES_DIR / f"figureqa_{row_idx:06d}.png"
    row["images"][0].convert("RGB").save(path, format="PNG")
    records.append({
        "id":            f"figureqa-{row_idx:06d}",
        "question":      q,
        "choices":       None,
        "answer":        a,
        "question_type": "factoid",
        "image":         str(path),
    })

DATASET_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2))
print(f"first {len(records)} charts (one question each) → {DATASET_JSON}")


In [ ]:
# ── sanity: same RoSE config as the pilot + locate the routing pool ──
import rose_chartqapro as rose
assert (rose.M_PATHS, rose.K_DEMONSTRATIONS, rose.LAMBDA, rose.TEMPERATURE) \
       == (5, 3, 1.2, 0.8), "rose_chartqapro defaults drifted from the sweep config"
assert not rose.PAPER_FAITHFUL
rose.IMAGES_DIR = str(IMAGES_DIR)
print(f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  T={rose.TEMPERATURE}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

# The routing pool: prefer the repo copy, fall back to an attached dataset.
POOL_FILE = CODE_DIR / "chartqapro/results/pilot/routing_pool.json"
if not POOL_FILE.exists():
    hits = sorted(Path("/kaggle/input").glob("*/routing_pool.json")) if ON_KAGGLE else []
    if hits:
        POOL_FILE = hits[0]
    else:
        raise SystemExit(
            "routing_pool.json not found.\n"
            "→ commit chartqapro/results/pilot/routing_pool.json to the branch\n"
            "  this notebook clones (built by routing/pool.py from the pilot\n"
            "  results), OR attach it as a Kaggle dataset.")
print(f"routing pool: {POOL_FILE}")

if RESUME_FROM:
    import shutil
    ckpt = OUT_FILE.with_name(OUT_FILE.stem + "_checkpoint.json")
    shutil.copy(RESUME_FROM, ckpt)
    print(f"resume checkpoint staged → {ckpt}")


In [ ]:
# ── STEP 1 — route every question (CPU, no model on the GPU yet) ─
# The kNN router embeds each question with the same all-mpnet encoder
# RoSE retrieves demonstrations with, scores each model on the K nearest
# pilot questions, and trusts the cheapest model clearing τ. Pilot
# questions re-encountered here are routed with themselves excluded
# (leave-one-out), so a decision is never a self-lookup.
import json
from routing.config import K_NEIGHBOURS, TAU as TAU_DEFAULT
from routing.routed_runner import route_dataset
from sentence_transformers import SentenceTransformer

k   = K   if K   is not None else K_NEIGHBOURS
tau = TAU if TAU is not None else TAU_DEFAULT

embedder = SentenceTransformer(rose.EMBED_MODEL)       # reused by RoSE below
_embed = lambda texts: embedder.encode(list(texts), convert_to_numpy=True)

records = json.loads(DATASET_JSON.read_text())
decisions = route_dataset(POOL_FILE, records, k=k, tau=tau,
                          embed_fn=_embed, out_path=DECISIONS_FILE)

# Projected wall-clock from the pilot's measured latencies (the 7B figure
# is an estimate — no per-question latency survives in the cached sweep).
# At 400 questions this usually exceeds one Kaggle session: plan to
# resume via RESUME_FROM (see the config cell).
EST_LAT = {"gemma3-4b": 115, "qwen3.5-4b": 260, "qwen2.5-vl-7b": 240}
proj = sum(decisions["share"][m] * len(records) * EST_LAT.get(m, 240)
           for m in decisions["share"])
print(f"\nprojected full-run time: {proj / 3600:.1f} h "
      f"({len(records)} questions; estimates, not promises)")
if proj > 8.5 * 3600:
    print("⚠ projection exceeds one Kaggle session — the checkpoint + "
          "RESUME_FROM flow in the config cell is how this finishes")
print(f"decisions → {DECISIONS_FILE}")


In [ ]:
# ── STEP 2 — RoSE with the routed model (GPU) ────────────────────
# Cheapest routed model first; each VLM is loaded once, answers its
# share, and is freed before the next loads. Checkpoint after EVERY
# question — if the session dies mid-run, nothing answered is lost
# (download the checkpoint from the Output pane and RESUME_FROM it).
# SESSION_BUDGET_H stops the run cleanly between questions so a headless
# "Save & Run All" version succeeds (a version KILLED at Kaggle's cap
# publishes nothing). DRY_RUN answers the first 4 questions (~15–25 min).
from routing.routed_runner import run_routed

results = run_routed(
    DATASET_JSON, decisions, OUT_FILE,
    quant=QUANT, embedder=embedder,
    limit=4 if DRY_RUN else None,
    stop_after_s=SESSION_BUDGET_H * 3600 if SESSION_BUDGET_H else None,
)


In [ ]:
# ── report ───────────────────────────────────────────────────────
from routing.config import MODELS

ok = [r for r in results if r.get("method") != "error"]
err = len(results) - len(ok)
acc = sum(r.get("is_correct", False) for r in ok) / max(len(ok), 1)
lat = [r["latency_s"] for r in ok if r.get("latency_s")]
params = [MODELS[r["model_key"]]["params_b"] for r in ok if r.get("model_key") in MODELS]

print(f"answered {len(ok)}   errors {err}")
print(f"routed accuracy : {acc * 100:5.1f}%")
if lat:
    print(f"mean latency    : {sum(lat) / len(lat):6.1f} s/question")
if params:
    print(f"mean params     : {sum(params) / len(params):6.2f} B/question "
          f"(always-7B would be {MODELS['qwen2.5-vl-7b']['params_b']:.2f} B)")

print(f"\n{'model':<16} {'routed':>7} {'answered':>9} {'accuracy':>9}")
for m in MODELS:
    share = decisions["share"].get(m, 0)
    sub = [r for r in ok if r.get("model_key") == m]
    a = sum(r.get("is_correct", False) for r in sub) / len(sub) * 100 if sub else None
    print(f"{m:<16} {share * 100:6.1f}% {len(sub):>9} "
          f"{(f'{a:8.1f}%' if a is not None else '       —')}")

print(f"\ndownload from the Output tab: {OUT_FILE.name}, "
      f"{OUT_FILE.stem}_meta.json, {OUT_FILE.stem}_checkpoint.json, "
      f"{DECISIONS_FILE.name}")


## After the session

**Headless chaining (Save & Run All):** with `SESSION_BUDGET_H` set, each
committed version runs ~8 h, stops cleanly, and PUBLISHES its outputs.
To chain sessions: Add Input → Your Work → this notebook (attaches the
latest successful version's output), set
`RESUME_FROM = "/kaggle/input/<notebook-slug>/results/routed/routed_<dataset>_checkpoint.json"`,
then Save & Run All again. Repeat until the report cell (or the meta
file's `"complete": true`) says every question is answered. Re-pin the
input to the newest version before each re-run.

1. Download from the Output pane: `routed_<dataset>.json`, `_meta.json`,
   `_checkpoint.json`, and `routing_decisions_<dataset>.json`.
2. **If the run is unfinished** (a 400-question run usually spans 2–3
   sessions): upload `routed_<dataset>_checkpoint.json` as a Kaggle
   dataset, set `RESUME_FROM` in the config cell, and run the notebook
   again — answered questions are skipped, nothing is redone.
3. When finished, drop the result + decisions + meta files into
   `chartqapro/results/routed/` in the repo. The meta file records
   per-model accuracy, routing share, mean params/question, resolved
   model revisions and library versions — the routed-vs-always-7B
   comparison for the report comes straight from it.
4. Quota bookkeeping: ~12–27 GPU-h per notebook at N=400 depending on
   the routing mix; the set of three is ~2–3 weeks of the 30 h quota.
   Run the notebooks in series. The first-N prefix is deterministic, so
   lowering or raising `N_QUESTIONS` later keeps all answered questions
   valid.
